# 24 · A told model for 60 fps at a 1280×720 stream on a GTX 1660 Ti

Notebook 23's told 24×2 graph is what desktop-streamer runs. On the GTX 1660 Ti a 1280×720 stream - 16 tiles of 328×188,
drawn at 2560×1440 - takes it 34.6 ms a frame on the benchmark page and 36.8 ms in the client, about 27 fps. The
reference (`reference_60fps.md`) is **60 fps with 20% to spare: 13.3 ms a frame in the client, which leaves the graph
9.2 ms** at 16 tiles, float16 with float32 at both ends. That is 3.8× less than it takes.

This notebook makes a model that fits, as a **drop-in** for the client's graph: the same `input` [N, 3, H, W] float32,
`condition` [3] (slice QP / 51, loop filter, I frame), `output` [N, 3, 2H, 2W], reaching no further than the client's
4 px halo, with a float16 twin. Nothing in desktop-streamer changes.

**Speed is predicted, not measured, on the 1660 Ti.** This computer has an RTX 3090. Every layer of the 1660 Ti's
cost table is measured again here on the 3090 and the ratio of the two taken per kind of kernel; a graph's 1660 Ti time
is its 3090 kernels times those ratios. A model is accepted at **8.3 ms predicted**, 10% under the 9.2 ms, because the
new kernels it uses have no measurement on the 1660 Ti.

1. **The budget** - where 9.2 ms goes today.
2. **What a layer costs on each card** - the calibration.
3. **Candidates** - what fits.
4. **Data, training, scores** - notebook 22's patches and recipe.
5. **Export** - the client's two graphs, checked as notebook 23 checks its own.

Needs: notebook 21 (`qp_slice_pairs/`) and notebook 22 (`checkpoints/upscale_qp_slice.pt`). Writes each training to
`checkpoints/fast_runs/`, the chosen model to `checkpoints/upscale_fast.pt`, timing graphs to `checkpoints/fast_probe/`,
and the graphs to `dist/upscale/client/upscale_fast.onnx` and `upscale_fast_fp16.onnx`.

In [1]:
"""Settings."""

import copy
import os
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, sources, training, webexport
from helpers.training import DEVICE

PAIRS_DIR = sources.ROOT / "upscale" / "qp_slice_pairs"           # notebook 21
RUN_DIR = training.CHECKPOINT_DIR / "fast_runs"                    # each training, saved every epoch
CHECKPOINT = training.CHECKPOINT_DIR / "upscale_fast.pt"           # the chosen model
TOLD = training.CHECKPOINT_DIR / "upscale_qp_slice.pt"             # notebook 22's told 24x2, the graph that ships
PROBE_DIR = training.CHECKPOINT_DIR / "fast_probe"                 # graphs to time on the benchmark page
CLIENT_DIR = webexport.MODELS_DIR / "client"                       # what the client's media/models/ takes

STEP = (180, 320)                    # the client's tile step (height, width)
HALO = 4                             # the client's halo (stream-enhance.js), which no model may reach past
TILE = (STEP[0] + 2 * HALO, STEP[1] + 2 * HALO)
TILES = 16                           # a 1280x720 stream
QP_MAX = 51
CONDITIONS = ("qp", "deblocked", "intra")
BUDGET_MS = 9.2                      # reference_60fps.md: the graph alone, float16, 16 tiles, on the 1660 Ti
BAR_MS = 8.3                         # 10% under it, since the 1660 Ti is predicted from the 3090
BATCH_SIZE, LEARNING_RATE, MAX_EPOCHS = 32, 1e-3, 500
OPSET = 17

training.seed_everything(0)

## 1. The budget

At 16 tiles the graph may take 9.2 ms on the benchmark page, and on the 1660 Ti a page time is its kernels plus about
13% (34.6 ms against 30.4 ms of kernels for the told model). That is **0.51 ms of kernels a tile**. In
`reference_60fps.md`'s table, what the told graph spends a tile around its network is already most of it:

| | ms a tile |
|---|---|
| the bicubic skip, a 5×5 convolution 3 → 12 | 0.17 |
| `Concat` of the luma to 12, `DepthToSpace` ×2, the output `Transpose`, the two `Cast`s | 0.15 |
| the `Add` of skip and correction | 0.02 |
| **left for the network** | **0.17** |

A 3×3 convolution at the LR pixels costs what its *input* width says (0.10 ms from the picture, 0.42 from 24 channels),
so 0.17 ms is a stem and a 1×1. Two things change that here:

- **The body runs at a quarter of the pixels.** A stride-2 stem makes one feature vector for every 2×2 LR cell; a
  3×3 there reads 2 LR px each way; a 1×1 tail writes the cell's 16 HR luma values and one `DepthToSpace` by 4 makes
  the picture. The same multiply-adds cover a 2× wider receptive field, and in ORT Web a wide layer over few pixels
  is cheaper than a narrow one over many.
- **The skip and the output path shrink.** Bicubic for a 2×2 cell is one 6×6 stride-2 convolution, 3 → 48
  (`models.QuarterBicubicSkip`, exact); without its outer ring of taps, which weigh 0.023 in bicubic's kernel, it is
  a 4×4 (`models.truncated_quarter_skip`) at half the cost, and frozen its colour is bicubic's to 0.01 dB. The tail's
  rows are written three times instead of a `Concat`.

## 2. What a layer costs on each card

Every row of `reference_60fps.md`'s layer table is one layer alone, float16, 16 tiles a run, and the time of its own
kernels. `probe_graphs` writes the same graphs; timed on the 3090 they give a ratio per kind of kernel. The 1660 Ti
column is that table's (2026-11-12, midpoints of its ranges); the 3090 column was measured 2026-11-13 on the
benchmark page (Chrome 157, ORT Web 1.29.0, `mode=reuse`, GPU-resident, the profiled time of the layer's own kernels,
which the page now posts as `gpuByKernel`).

To measure again: run this cell, then

    uv run src/upscale/benchmark/main.py --models src/upscale/checkpoints/fast_probe/layers --port 8765 --results layers.jsonl

and open `http://127.0.0.1:8765/?auto=all&backend=webgpu&data=gpu&mode=reuse&seconds=2` in Chrome started with
`--enable-unsafe-webgpu --enable-webgpu-developer-features`.

In [2]:
def write_graph(module, inputs, path, label):
    """`module` exported at a fixed shape, then float16 inside and float32 at both ends, as the client runs a graph."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path32 = path.with_name(path.stem + "_f32.onnx")
    names = [f"input{i}" if i else "input" for i in range(len(inputs))]
    torch.onnx.export(module.eval(), tuple(inputs), str(path32), opset_version=OPSET, input_names=names,
                      output_names=["output"], dynamo=False)
    half = webexport.to_float16(webexport.describe(path32), path.name, models_dir=path.parent, keep_io_types=True)
    path32.unlink()
    webexport.write_metadata(half["path"], {"label": label, "step": STEP[1], "step_h": STEP[0], "halo": HALO,
                                            "batch": TILES})
    return half


class Layer(nn.Module):
    def __init__(self, conv, act=True):
        super().__init__()
        self.conv, self.act = conv, (nn.LeakyReLU(0.1) if act else nn.Identity())

    def forward(self, x):
        return self.act(self.conv(x))


class AddProbe(nn.Module):
    """An `Add` of two real tensors: the input and a 1x1 of it."""

    def __init__(self, channels):
        super().__init__()
        self.point = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        return x + self.point(x)


LR, QUARTER = TILE, (TILE[0] // 2, TILE[1] // 2)
# name: (layer, input channels, input size); the 1660 Ti table's rows first, then what is new here
PROBES = {
    "3x3, 3 in -> 24": (Layer(nn.Conv2d(3, 24, 3, padding=1)), 3, LR),
    "3x3, 4 in": (Layer(nn.Conv2d(4, 24, 3, padding=1)), 4, LR),
    "3x3, 8 in": (Layer(nn.Conv2d(8, 24, 3, padding=1)), 8, LR),
    "3x3, 12 in": (Layer(nn.Conv2d(12, 24, 3, padding=1)), 12, LR),
    "3x3, 16 in": (Layer(nn.Conv2d(16, 16, 3, padding=1)), 16, LR),
    "3x3, 24 in -> 24": (Layer(nn.Conv2d(24, 24, 3, padding=1)), 24, LR),
    "3x3, 24 in -> 4": (Layer(nn.Conv2d(24, 4, 3, padding=1), act=False), 24, LR),
    "3x3, 32 in -> 32": (Layer(nn.Conv2d(32, 32, 3, padding=1)), 32, LR),
    "1x1, 24": (Layer(nn.Conv2d(24, 24, 1)), 24, LR),
    "3x3 depthwise, 24": (Layer(nn.Conv2d(24, 24, 3, padding=1, groups=24)), 24, LR),
    "5x5 skip, 3 -> 12": (Layer(nn.Conv2d(3, 12, 5, padding=2, bias=False), act=False), 3, LR),
    "Add, 24": (AddProbe(24), 24, LR),
    "Add, 12": (AddProbe(12), 12, LR),
    "stem 4x4/2, 3 -> 24": (Layer(nn.Conv2d(3, 24, 4, stride=2, padding=1)), 3, LR),
    "stem 4x4/2, 3 -> 32": (Layer(nn.Conv2d(3, 32, 4, stride=2, padding=1)), 3, LR),
    "skip 6x6/2, 3 -> 48 (exact)": (Layer(nn.Conv2d(3, 48, 6, stride=2, padding=2, bias=False), act=False), 3, LR),
    "skip 4x4/2, 3 -> 48": (Layer(nn.Conv2d(3, 48, 4, stride=2, padding=1, bias=False), act=False), 3, LR),
    "quarter 3x3, 24": (Layer(nn.Conv2d(24, 24, 3, padding=1)), 24, QUARTER),
    "quarter 3x3, 32": (Layer(nn.Conv2d(32, 32, 3, padding=1)), 32, QUARTER),
    "quarter 3x3, 48": (Layer(nn.Conv2d(48, 48, 3, padding=1)), 48, QUARTER),
    "quarter 1x1, 32": (Layer(nn.Conv2d(32, 32, 1)), 32, QUARTER),
    "quarter 1x1, 32 -> 48": (Layer(nn.Conv2d(32, 48, 1), act=False), 32, QUARTER),
    "quarter 3x3 depthwise, 64": (Layer(nn.Conv2d(64, 64, 3, padding=1, groups=64)), 64, QUARTER),
}


def probe_graphs(folder):
    folder.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(0)
    for index, (name, (layer, channels, size)) in enumerate(PROBES.items()):
        write_graph(layer, [torch.rand(TILES, channels, *size)], folder / f"layer{index:02d}.onnx", name)


probe_graphs(PROBE_DIR / "layers")

# us a tile: (kind of kernel, 1660 Ti, 3090), the layer's own kernels; None where the 1660 Ti was not measured
LAYER_US = {
    "3x3, 3 in -> 24": ("conv", 105, 23.8),
    "3x3, 4 in": ("conv", 130, 27.5),
    "3x3, 8 in": ("conv", 175, 39.4),
    "3x3, 12 in": ("conv", 215, 51.1),
    "3x3, 16 in": ("conv", 300, 62.9),
    "3x3, 24 in -> 24": ("conv", 415, 86.9),
    "3x3, 24 in -> 4": ("conv", 400, 85.9),
    "3x3, 32 in -> 32": ("conv wide", 570, 91.3),
    "1x1, 24": ("matmul", 75, 11.0),
    "3x3 depthwise, 24": ("grouped", 95, 56.2),
    "5x5 skip, 3 -> 12": ("conv", 170, 42.9),
    "Add, 24": ("add", 34, 10.4),
    "Add, 12": ("add", 19, 5.3),
    # the told graph's Concat, DepthToSpace, output Transpose and Casts, measured inside it on the 3090
    "the output path": ("move", 150, 32.6),
    "stem 4x4/2, 3 -> 24": ("conv", None, 6.3),
    "stem 4x4/2, 3 -> 32": ("conv", None, 6.1),
    "skip 6x6/2, 3 -> 48 (exact)": ("conv", None, 27.1),
    "skip 4x4/2, 3 -> 48": ("conv", None, 12.4),
    "quarter 3x3, 24": ("conv", None, 22.6),
    "quarter 3x3, 32": ("conv wide", None, 27.5),
    "quarter 3x3, 48": ("conv wide", None, 89.1),
    "quarter 1x1, 32": ("matmul", None, 3.5),
    "quarter 1x1, 32 -> 48": ("matmul", None, 6.1),
    "quarter 3x3 depthwise, 64": ("grouped", None, 38.2),
}

print(f"{'layer':<30}{'kind':<11}{'1660 Ti':>9}{'3090':>8}{'ratio':>7}")
for name, (kind, slow, fast) in LAYER_US.items():
    ratio = f"{slow / fast:7.1f}" if slow else ""
    print(f"{name:<30}{kind:<11}{(slow or ''):>9}{fast:8.1f}{ratio}")

/tmp/ipykernel_80287/2367683993.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(module.eval(), tuple(inputs), str(path32), opset_version=OPSET, input_names=names,


layer                         kind         1660 Ti    3090  ratio
3x3, 3 in -> 24               conv             105    23.8    4.4
3x3, 4 in                     conv             130    27.5    4.7
3x3, 8 in                     conv             175    39.4    4.4
3x3, 12 in                    conv             215    51.1    4.2
3x3, 16 in                    conv             300    62.9    4.8
3x3, 24 in -> 24              conv             415    86.9    4.8
3x3, 24 in -> 4               conv             400    85.9    4.7
3x3, 32 in -> 32              conv wide        570    91.3    6.2
1x1, 24                       matmul            75    11.0    6.8
3x3 depthwise, 24             grouped           95    56.2    1.7
5x5 skip, 3 -> 12             conv             170    42.9    4.0
Add, 24                       add               34    10.4    3.3
Add, 12                       add               19     5.3    3.6
the output path               move             150    32.6    4.6
stem 4x4/2

The ratio is not one number. Convolutions from up to 24 channels run 4.0-4.8× slower on the 1660 Ti, 1×1s (a
`MatMul` kernel) about 7×, depthwise convolutions about 2×, `Add` 3.3-3.6×. The 3090 hides a wider input that the 1660
Ti does not: 32 channels in cost the 3090 5% more than 24, and the 1660 Ti 37% more - a ratio of 6.2. So a
prediction charges each kernel by its kind, two ways:

- **central:** the typical ratio of each kind;
- **conservative:** the largest ratio of each kind, and 6.2 for a convolution reading 28 or more channels.

Checked on the one whole graph measured on both cards: the told 24×2 at 16 tiles is 34.6 ms on the 1660 Ti, and
from its 3090 kernels the central prediction is 36.4 and the conservative one 37.7. Both err high, by 5% and 9%.

In [3]:
RATIO = {"conv": (4.6, 4.8), "conv wide": (4.6, 6.2), "matmul": (6.9, 7.0), "grouped": (2.1, 2.4),
         "add": (3.5, 3.6), "move": (4.6, 4.6)}   # 1660 Ti over 3090: (central, conservative)
OVERHEAD = 1.13                                    # the 1660 Ti's page time over its kernels


def predict(kernels, tiles=TILES):
    """A graph's 1660 Ti page time in ms, (central, conservative), from its 3090 kernels in us a tile by kind."""
    return tuple(OVERHEAD * tiles * sum(us * RATIO[kind][which] for kind, us in kernels.items()) / 1000
                 for which in (0, 1))


# the told 24x2, float16, 16 tiles, on the 3090 (2026-11-13): page ms, kernels in us a tile by kind
TOLD_3090 = (6.95, {"move": 35.1, "conv": 342.6, "matmul": 25.2, "add": 28.5})
central, conservative = predict(TOLD_3090[1])
print(f"told 24x2: 3090 {TOLD_3090[0]:.2f} ms; 1660 Ti predicted {central:.1f} (conservative {conservative:.1f}), "
      f"measured 34.6")

told 24x2: 3090 6.95 ms; 1660 Ti predicted 36.4 (conservative 37.7), measured 34.6


## 3. Candidates

`models.FastLuma` is the body: a stride-2 stem (4×4, reaching 1 LR px past its cell), a residual block at a quarter of
the pixels (3×3 then 1×1, added back: 2 px more), a 1×1 tail to the cell's 16 luma values, a leaky ReLU (notebook 16:
a narrow body loses channels to a plain one), and the told shifts after the stem and in the block, folded into their
biases for the client by `models.FastLumaShipped`. With `layout="lr"` it is the same at the LR pixels, the control.

Each candidate is exported at 16 tiles with random weights and timed on the 3090 (2026-11-13, as above: page time and
its kernels by kind). Two blocks after a 2×2 stem reach 4 px, the halo; a third, or a second after the 4×4 stem, would
reach past it, which a tile cannot see.

In [4]:
CANDIDATES = {   # name: FastLuma arguments
    "q20": dict(width=20, stem=4, blocks=1, skip="linear"),
    "q24": dict(width=24, stem=4, blocks=1, skip="linear"),
    "q24 frozen skip": dict(width=24, stem=4, blocks=1, skip="truncated"),
    "q28": dict(width=28, stem=4, blocks=1, skip="linear"),
    "q32": dict(width=32, stem=4, blocks=1, skip="linear"),
    "q24 exact skip": dict(width=24, stem=4, blocks=1, skip="bicubic"),
    "q24 + 1x1 block": dict(width=24, stem=4, blocks=1, points=1, skip="linear"),
    "q16, 2 blocks": dict(width=16, stem=2, blocks=2, skip="linear"),
    "q24, 2 blocks": dict(width=24, stem=2, blocks=2, skip="linear"),
    "lr16": dict(width=16, stem=3, blocks=0, layout="lr"),
    "lr24": dict(width=24, stem=3, blocks=0, layout="lr"),
}

# 3090, 16 tiles, float16 (2026-11-13): page ms, kernels in us a tile by kind
MEASURED_3090 = {
    "q20": (1.35, {"move": 27.5, "conv": 38.2, "matmul": 9.4, "add": 8.3}),
    "q24": (1.42, {"move": 26.1, "conv": 43.4, "matmul": 10.6, "add": 8.0}),
    "q24 frozen skip": (1.42, {"move": 26.1, "conv": 43.4, "matmul": 10.6, "add": 8.0}),   # q24's graph, other weights
    "q28": (1.49, {"move": 26.1, "conv": 18.3, "conv wide": 25.5, "matmul": 12.6, "add": 9.6}),
    "q32": (1.52, {"move": 25.7, "conv": 19.2, "conv wide": 28.6, "matmul": 12.2, "add": 8.9}),
    "q24 exact skip": (1.67, {"move": 27.4, "conv": 55.4, "matmul": 10.6, "add": 8.0}),
    "q24 + 1x1 block": (1.58, {"move": 26.9, "conv": 42.6, "matmul": 15.6, "add": 10.8}),
    "q16, 2 blocks": (1.61, {"move": 25.7, "conv": 51.3, "matmul": 12.2, "add": 9.1}),
    "q24, 2 blocks": (1.86, {"move": 27.4, "conv": 61.8, "matmul": 12.7, "add": 10.8}),
    "lr16": (1.82, {"move": 27.4, "conv": 67.7, "matmul": 11.9, "add": 6.5}),
    "lr24": (1.83, {"move": 28.0, "conv": 68.0, "matmul": 11.6, "add": 5.3}),
}

print("the exact quarter skip is bicubic: largest difference", models.check_quarter_skip_is_bicubic())
torch.manual_seed(0)
probe = torch.rand(TILES, 3, *TILE)
print(f"{'candidate':<18}{'halo':>5}{'MAC/LR px':>11}{'3090 ms':>9}{'1660 Ti ms':>12}{'conservative':>14}")
for name, arguments in CANDIDATES.items():
    model = models.FastLuma(**arguments).eval()
    assert model.halo <= HALO, f"{name} reaches {model.halo} px past its tile"
    for parameter in model.parameters():
        if parameter.requires_grad:
            nn.init.normal_(parameter, std=0.1)   # random weights: only the shapes matter for timing
    write_graph(models.FastLumaShipped(model), [probe, torch.rand(3)],
                PROBE_DIR / "candidates" / f"{name.replace(' ', '_').replace(',', '').replace('+', 'p')}.onnx",
                f"candidate {name}")
    # a weight is a multiply-add for every cell (4 LR px at a quarter of the pixels); the skip and shifts left out
    weights = sum(p.numel() for n, p in model.named_parameters() if p.dim() == 4 and not n.startswith(("shifts", "skip")))
    macs = weights / (4 if model.layout == "quarter" else 1)
    page, kernels = MEASURED_3090[name]
    central, conservative = predict(kernels)
    flag = "" if conservative <= BAR_MS else ("  over the bar" if conservative <= BUDGET_MS else "  over the budget")
    print(f"{name:<18}{model.halo:5}{macs:11,.0f}{page:9.2f}{central:12.1f}{conservative:14.1f}{flag}")

the exact quarter skip is bicubic: largest difference 0.0
candidate          halo  MAC/LR px  3090 ms  1660 Ti ms  conservative
q20                   3      1,320     1.35         7.2           7.3
q24                   3      1,824     1.42         7.6           7.8


/tmp/ipykernel_80287/2367683993.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(module.eval(), tuple(inputs), str(path32), opset_version=OPSET, input_names=names,


q24 frozen skip       3      1,824     1.42         7.6           7.8
q28                   3      2,408     1.49         8.0           8.8  over the bar
q32                   3      3,072     1.52         8.2           9.1  over the bar
q24 exact skip        3      1,824     1.67         8.7           8.9  over the bar
q24 + 1x1 block       3      2,112     1.58         8.4           8.6  over the bar
q16, 2 blocks         4      1,392     1.61         8.5           8.7  over the bar
q24, 2 blocks         4      3,048     1.86         9.7          10.0  over the budget
lr16                  2        496     1.82         9.8          10.1  over the budget
lr24                  2        744     1.83         9.8          10.0  over the budget


Only the quarter-pixel layout fits. At the LR pixels a stem and a 1×1 tail are already 9.8 ms, central: the exact 5×5
skip and the output path at full size are most of it. At a quarter of the pixels a 24-wide body predicts 7.6 ms
central and 7.8 conservative with the 4×4 skip; the exact skip adds 1.1 ms, a 1×1 block 0.8, a second 3×3 block
2.1, and widening to 28 or 32 runs into the 1660 Ti's wide-input ratio.

Trained below: **q20** and **q24**, which fit, with the 4×4 skip trained; **q24 frozen skip**, the same graph with the
4×4 frozen at its start; **q32**, to see what width would buy; and **q24 exact skip**, to see what the 4×4 costs.

## 4. Data

Notebook 22's patches and conditions: every patch carries the slice QP / 51, the loop filter and I frame.

In [5]:
manifest = training.load_manifest(PAIRS_DIR)
training.describe(manifest)
SCALE = manifest["scale"]
ENCODERS = manifest["encoders"]
trim = training.trimmer(manifest)
assert manifest["halo"] >= HALO


class WithConditions(Dataset):
    """A split's pairs as ((LR, condition (3, 1, 1)), HR), and each patch's labels as arrays."""

    def __init__(self, split, augment):
        self.pairs = patches.PatchPairs(PAIRS_DIR, split, manifest, augment=augment)
        files = manifest["splits"][split]
        self.labels = {name: np.load(PAIRS_DIR / files[name]) for name in ("qp", "deblocked", "intra", "encoder")}
        values = np.stack([self.labels["qp"] / QP_MAX, self.labels["deblocked"], self.labels["intra"]], axis=1)
        self.conditions = torch.from_numpy(values.astype(np.float32))[:, :, None, None]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        lr, hr = self.pairs[index]
        return (lr, self.conditions[index]), hr

    def where(self, encoder=None, intra=None):
        keep = np.ones(len(self), dtype=bool)
        if encoder is not None:
            keep &= self.labels["encoder"] == ENCODERS.index(encoder)
        if intra is not None:
            keep &= self.labels["intra"] == intra
        return np.flatnonzero(keep)


def loader(dataset, indices=None, train=False):
    kept = dataset if indices is None else Subset(dataset, indices)
    return DataLoader(kept, batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train)


train_set, val_set = WithConditions("train", augment=True), WithConditions("val", augment=False)
print(f"train {len(train_set)} patches, val {len(val_set)} patches")

device  cuda NVIDIA GeForce RTX 3090
scale   2
patch   72 px LR in (64 px + a halo of 4) -> 128 px HR out, 1 frames each
train 34364 patches, val 9628 patches


## 5. Train

Notebook 22's recipe: Adam from 1e-3, batch 32, `metrics.ssim_l1_loss`, the epoch with the best SSIM-Y on the
validation patches kept, the rate halved on a plateau until it converges. One seed each; the two that fit and score
best get a second. Run the cell again after an interruption and it carries on; a finished training just loads.

In [6]:
RUNS = {"q24": (0, 1), "q20": (0, 1), "q32": (0,), "q24 exact skip": (0,), "q24 frozen skip": (0, 1)}   # name: seeds


def train(name, seed):
    torch.manual_seed(seed)
    model = models.FastLuma(**CANDIDATES[name]).to(DEVICE)
    torch.manual_seed(seed)
    run = f"{name.replace(' ', '_')}_seed{seed}"
    history, best = training.train(model, loader(train_set, train=True), loader(val_set), trim, MAX_EPOCHS,
                                   learning_rate=LEARNING_RATE, loss_fn=metrics.ssim_l1_loss, select="ssim",
                                   checkpoint=RUN_DIR / f"{run}.pt", verbose=0)
    print(f"{run:<26} best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f}")
    return model.eval(), best


trained, bests = {}, {}
for seed in (0, 1):
    for name, seeds in RUNS.items():
        if seed in seeds:
            trained[name, seed], bests[name, seed] = train(name, seed)

q24_seed0                  best epoch  71 of  89, ssim-y 0.9212
q20_seed0                  best epoch  92 of  98, ssim-y 0.9199
q32_seed0                  best epoch 106 of 112, ssim-y 0.9229
q24_exact_skip_seed0       best epoch 106 of 112, ssim-y 0.9213
q24_frozen_skip_seed0      best epoch  92 of 104, ssim-y 0.9212
q24_seed1                  best epoch 120 of 126, ssim-y 0.9215
q20_seed1                  best epoch  95 of 101, ssim-y 0.9196
q24_frozen_skip_seed1      best epoch 120 of 126, ssim-y 0.9212


## 6. Scores

SSIM-Y and luma PSNR on the 9,628 validation patches, absolute and over bicubic, beside notebook 22's told 24×2 -
the graph that ships - and by encoder and frame type, and the chroma PSNR (U and V) overall. The network adds the same
value to R, G and B, which leaves U and V alone, so a model's colour is its skip's: bicubic's for the told 24×2 and
any frozen skip. A candidate with two seeds is their mean.

In [7]:
class Bicubic(nn.Module):
    def forward(self, x, condition=None):
        return models.bicubic(x, SCALE)


saved = training.load_checkpoint(TOLD)
told = models.ToldLuma(scale=saved["scale"], **saved["arguments"]).to(DEVICE)
told.load_state_dict(saved["state_dict"])
told.eval()

columns = {"bicubic": [Bicubic()], "told 24x2 (ships)": [told],
           **{name: [trained[name, seed] for seed in seeds] for name, seeds in RUNS.items()}}
GROUPS = [("all", {}), *[(f"{e} {kind}", {"encoder": e, "intra": kind == "I"}) for e in ENCODERS for kind in "PI"]]

table = {}
for label, where in GROUPS:
    data = loader(val_set, val_set.where(**where))
    table[label] = {}
    for name, runs in columns.items():
        each = [training.evaluate(model, data, trim) for model in runs]
        table[label][name] = {key: float(np.mean([s[key] for s in each])) for key in ("ssim", "y", "u", "v")}
        table[label][name]["gap"] = abs(each[0]["ssim"] - each[1]["ssim"]) if len(each) > 1 else None

print("absolute, all patches: SSIM-Y / luma dB, and U / V dB")
for name in columns:
    row = table["all"][name]
    gap = f"   seeds {row['gap']:.4f} apart" if row["gap"] is not None else ""
    print(f"  {name:<20}{row['ssim']:.4f} / {row['y']:.2f}    {row['u']:.2f} / {row['v']:.2f}{gap}")
print()
print("over bicubic: SSIM-Y / luma dB")
print(f"{'':<12}" + "".join(f"{name:>20}" for name in columns if name != "bicubic"))
for label, _ in GROUPS:
    base = table[label]["bicubic"]
    cells = [f"{table[label][n]['ssim'] - base['ssim']:+.4f} / {table[label][n]['y'] - base['y']:+.2f}"
             for n in columns if n != "bicubic"]
    print(f"{label:<12}" + "".join(f"{cell:>20}" for cell in cells))

absolute, all patches: SSIM-Y / luma dB, and U / V dB
  bicubic             0.9045 / 33.55    43.99 / 43.93
  told 24x2 (ships)   0.9289 / 35.07    44.03 / 43.97
  q24                 0.9214 / 34.43    43.04 / 43.23   seeds 0.0003 apart
  q20                 0.9197 / 34.34    43.06 / 43.22   seeds 0.0003 apart
  q32                 0.9229 / 34.55    43.03 / 43.26
  q24 exact skip      0.9213 / 34.51    44.00 / 43.95
  q24 frozen skip     0.9212 / 34.47    44.00 / 43.94   seeds 0.0001 apart

over bicubic: SSIM-Y / luma dB
               told 24x2 (ships)                 q24                 q20                 q32      q24 exact skip     q24 frozen skip
all              +0.0243 / +1.52     +0.0168 / +0.87     +0.0152 / +0.78     +0.0184 / +1.00     +0.0167 / +0.96     +0.0167 / +0.92
x264 P           +0.0302 / +1.56     +0.0225 / +0.94     +0.0207 / +0.85     +0.0242 / +1.06     +0.0223 / +1.01     +0.0222 / +0.97
x264 I           +0.0288 / +1.66     +0.0223 / +1.00     +0.0208 / +0.89  

## 7. Save and export

The candidate with the best SSIM-Y among those predicted at or under 8.3 ms that keep bicubic's colour (U and V
within 0.1 dB of it), its better seed. Exported as notebook 23
exports the told 24×2: the classic exporter, opset 17, `input` and `condition` in, `output` out, `N`, `H` and `W`
symbolic for the client, a static tile graph for the benchmark page, and the float16 twin with float32 at both ends.

In [8]:
base = table["all"]["bicubic"]
fits = [name for name in RUNS if predict(MEASURED_3090[name][1])[1] <= BAR_MS
        and min(table["all"][name]["u"] - base["u"], table["all"][name]["v"] - base["v"]) >= -0.1]
NAME = max(fits, key=lambda name: table["all"][name]["ssim"])
seed = max(RUNS[NAME], key=lambda s: bests[NAME, s]["score"])
best = trained[NAME, seed]
print(f"chosen: {NAME}, seed {seed} (fits: {', '.join(fits)})")
training.save_checkpoint(CHECKPOINT, best, architecture="FastLuma", arguments=CANDIDATES[NAME], name=NAME,
                         told=list(CONDITIONS), qp_max=QP_MAX, qp="slice header", scale=SCALE, halo=best.halo,
                         loss="ssim_l1_loss", select="ssim", seed=seed, best_epoch=bests[NAME, seed]["epoch"],
                         predicted_1660ti_ms=list(predict(MEASURED_3090[NAME][1])),
                         val={label: {n: {key: row[n][key] for key in ("ssim", "y", "u", "v")} for n in row}
                              for label, row in table.items()})

saved = training.load_checkpoint(CHECKPOINT)
fast = models.FastLuma(**saved["arguments"]).to(DEVICE)
fast.load_state_dict(saved["state_dict"])
fast = fast.cpu().eval()
for frame in ((30 / QP_MAX, 0, 0), (36 / QP_MAX, 1, 1)):
    training.assert_same_model(best.baked(frame), fast.baked(frame))
print("reloaded model is identical")

chosen: q24 frozen skip, seed 0 (fits: q24 frozen skip)
saved /home/henrik/projects/desktop-streamer-ai/src/upscale/checkpoints/upscale_fast.pt (46 KB)
reloaded model is identical


In [9]:
def condition_of(qp, deblocked, intra):
    """What the client feeds: the slice QP / 51, the loop filter, I frame."""
    return np.array([qp / QP_MAX, float(deblocked), float(intra)], dtype=np.float32)


def export(model, path, size, dynamic):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        copy.deepcopy(model).eval(), (torch.rand(1, 3, *size), torch.from_numpy(condition_of(30, 1, 0))), str(path),
        opset_version=OPSET, input_names=["input", "condition"], output_names=["output"],
        dynamic_axes=({"input": {0: "N", 2: "H", 3: "W"}, "output": {0: "N", 2: "H2", 3: "W2"}} if dynamic else None),
        dynamo=False)
    onnx.checker.check_model(onnx.load(str(path)))
    return path


SHORT = NAME.replace(" ", "_")
LABEL = f"told fast {NAME} (notebook 24)"
told_fields = {"told": ",".join(CONDITIONS), "qp_max": QP_MAX, "qp": "slice header",
               "condition": "slice QP / 51, loop filter 0 or 1, I frame 0 or 1"}
shipped = models.FastLumaShipped(fast).eval()

tile_path = export(shipped, webexport.MODELS_DIR / f"upscale_fast_{SHORT}_tile{STEP[1]}x{STEP[0]}.onnx", TILE, dynamic=False)
webexport.write_metadata(tile_path, {"label": LABEL, "step": STEP[1], "step_h": STEP[0], "halo": HALO, "batch": 1,
                                     **told_fields})
client_path = export(shipped, CLIENT_DIR / "upscale_fast.onnx", TILE, dynamic=True)
webexport.write_metadata(client_path, {"label": LABEL, "halo": HALO, "scale": SCALE, **told_fields})
half_path = webexport.to_float16(webexport.describe(client_path), "upscale_fast_fp16.onnx", models_dir=CLIENT_DIR,
                                 keep_io_types=True)["path"]

# the float16 client graph at a 1280x720 stream's 16 tiles, for the benchmark page (Run 8 of bottlenecks_gtx1660ti.md)
bench = onnx.load(str(half_path))
for value, dims in ((bench.graph.input[0], (TILES, 3, *TILE)), (bench.graph.output[0], (TILES, 3, TILE[0] * 2, TILE[1] * 2))):
    for dim, size in zip(value.type.tensor_type.shape.dim, dims):
        dim.dim_value = size
onnx.checker.check_model(bench)
bench_path = webexport.MODELS_DIR / f"upscale_fast_{SHORT}_fp16_tile{STEP[1]}x{STEP[0]}_b{TILES}.onnx"
onnx.save(bench, str(bench_path))
webexport.write_metadata(bench_path, {"label": f"{LABEL}, {TILES} tiles, float16", "step": STEP[1], "step_h": STEP[0],
                                      "halo": HALO, "batch": TILES, "precision": "float16", **told_fields})

for path in (tile_path, client_path, half_path, bench_path):
    print(webexport.summary_line(webexport.describe(path)))
    graph = onnx.load(str(path)).graph
    for value in [*graph.input, *graph.output]:
        dims = [d.dim_value if d.HasField("dim_value") else d.dim_param for d in value.type.tensor_type.shape.dim]
        print("   ", value.name, dims)

upscale_fast_q24_frozen_skip_tile320x180.onnx          44.4 KB   Addx2, Constantx3, Convx7, DepthToSpacex1, LeakyRelux2, Reshapex3   OUTSIDE BUDGET: Reshape
    input [1, 3, 188, 328]
    condition [3]
    output [1, 3, 376, 656]
upscale_fast.onnx                                      44.4 KB   Addx2, Constantx3, Convx7, DepthToSpacex1, LeakyRelux2, Reshapex3   OUTSIDE BUDGET: Reshape
    input ['N', 3, 'H', 'W']
    condition [3]
    output ['N', 3, 'H2', 'W2']
upscale_fast_fp16.onnx                                 24.9 KB   Addx2, Castx3, Constantx3, Convx7, DepthToSpacex1, LeakyRelux2, Reshapex3   OUTSIDE BUDGET: Cast, Reshape
    input ['N', 3, 'H', 'W']
    condition [3]
    output ['N', 3, 'H2', 'W2']
upscale_fast_q24_frozen_skip_fp16_tile320x180_b16.onnx   24.9 KB   Addx2, Castx3, Constantx3, Convx7, DepthToSpacex1, LeakyRelux2, Reshapex3   OUTSIDE BUDGET: Cast, Reshape
    input [16, 3, 188, 328]
    condition [3]
    output [16, 3, 376, 656]


/tmp/ipykernel_80287/2032635816.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


### Checks

Notebook 23's: both graphs against the model baked for one frame at six conditions, at the client's batch and at sizes
it never uses; the client graph with its dimensions written as numbers, as the client's WebGL path does; the float16
graph within an 8-bit step of the float32 one. Then two of this notebook's own:

- **No seams.** A whole 1280×720 frame cut into the client's tiles - a 320×180 step, a 4 px halo, the frame's edge
  pixels read again outside it (`planTiles` and the tile shader in `stream-enhance.js`) - through the client graph,
  against the model on the whole frame padded the same way. The kept regions must agree to float precision: the
  model reaches no further than the halo, and every tile's stride-2 grid lines up with the frame's.
- **The graph scores what the model scores** on the validation patches.

In [10]:
def run(session, x, condition):
    return session.run(None, {"input": x.astype(np.float32), "condition": condition})[0]


tile_session, client_session = webexport.session(tile_path), webexport.session(client_path)
worst = 0.0
for frame in ((0, 0, 0), (18, 1, 0), (30, 0, 0), (34, 1, 1), (42, 0, 1), (51, 1, 0)):
    condition = condition_of(*frame)
    baked = fast.baked(condition)
    x = np.random.rand(TILES, 3, *TILE).astype(np.float32)
    with torch.no_grad():
        expected = baked(torch.from_numpy(x)).numpy()
    worst = max(worst, float(np.abs(run(client_session, x, condition) - expected).max()))
    worst = max(worst, float(np.abs(run(tile_session, x[:1], condition) - expected[:1]).max()))
    for size in ((1, 3, 64, 48), (2, 3, 720 // 2 + 8, 1280 // 2 + 8)):   # any even size the client hands it
        y = np.random.rand(*size).astype(np.float32)
        with torch.no_grad():
            worst = max(worst, float(np.abs(run(client_session, y, condition) - baked(torch.from_numpy(y)).numpy()).max()))
print(f"both graphs compute the baked model for every condition: largest difference {worst:.1e}")
assert worst < 1e-4

fixed = onnx.load(str(client_path))
for value in [fixed.graph.input[0], fixed.graph.output[0]]:
    for dim, size in zip(value.type.tensor_type.shape.dim, (36, 3, TILE[0] * (SCALE if value.name == "output" else 1),
                                                            TILE[1] * (SCALE if value.name == "output" else 1))):
        dim.dim_value = size
onnx.checker.check_model(fixed)
fixed_session = ort.InferenceSession(fixed.SerializeToString(), providers=["CPUExecutionProvider"])
x, c = np.random.rand(36, 3, *TILE).astype(np.float32), condition_of(33, 1, 0)
assert np.abs(fixed_session.run(None, {"input": x, "condition": c})[0] - run(client_session, x, c)).max() < 1e-6
print("with its dimensions fixed, as on WebGL, it computes the same")

half_session = webexport.session(half_path)
worst = 0.0
for frame in ((0, 0, 0), (18, 1, 0), (30, 0, 0), (34, 1, 1), (42, 0, 1), (51, 1, 0)):
    condition = condition_of(*frame)
    x = np.random.rand(TILES, 3, *TILE).astype(np.float32)
    worst = max(worst, float(np.abs(run(half_session, x, condition) - run(client_session, x, condition)).max()))
print(f"the float16 graph is within {worst * 255:.2f}/255 of the float32 one")
assert worst < 1 / 255

both graphs compute the baked model for every condition: largest difference 5.7e-07
with its dimensions fixed, as on WebGL, it computes the same
the float16 graph is within 0.16/255 of the float32 one


In [11]:
def plan_tiles(width, height):
    """stream-enhance.js's planTiles: each tile's kept region (x, y, w, h) and its window's origin (ix, iy)."""
    tile_w, tile_h = min(width, STEP[1]) + 2 * HALO, min(height, STEP[0]) + 2 * HALO
    tiles = []
    for y in range(0, height, STEP[0]):
        h, iy = min(STEP[0], height - y), min(y - HALO, height + HALO - tile_h)
        for x in range(0, width, STEP[1]):
            w, ix = min(STEP[1], width - x), min(x - HALO, width + HALO - tile_w)
            tiles.append((x, y, w, h, ix, iy))
    return tile_w, tile_h, tiles


width, height = 1280, 720
frame = np.random.rand(3, height, width).astype(np.float32)
padded = np.pad(frame, ((0, 0), (HALO, HALO), (HALO, HALO)), mode="edge")   # the edge pixel, read again
tile_w, tile_h, tiles = plan_tiles(width, height)
windows = np.stack([padded[:, iy + HALO:iy + HALO + tile_h, ix + HALO:ix + HALO + tile_w] for *_, ix, iy in tiles])
condition = condition_of(30, 1, 0)
out = run(client_session, windows, condition)
tiled = np.zeros((3, height * SCALE, width * SCALE), dtype=np.float32)
for picture, (x, y, w, h, ix, iy) in zip(out, tiles):
    ox, oy = (x - ix) * SCALE, (y - iy) * SCALE
    tiled[:, y * SCALE:(y + h) * SCALE, x * SCALE:(x + w) * SCALE] = picture[:, oy:oy + h * SCALE, ox:ox + w * SCALE]
with torch.no_grad():
    whole = fast.baked(condition)(torch.from_numpy(padded[None])).numpy()[0]
whole = whole[:, HALO * SCALE:-HALO * SCALE, HALO * SCALE:-HALO * SCALE]
difference = float(np.abs(tiled - whole).max())
print(f"{len(tiles)} tiles of {tile_w}x{tile_h} against the whole frame: largest difference {difference:.1e}")
assert difference < 1e-4

16 tiles of 328x188 against the whole frame: largest difference 6.0e-07


In [12]:
class Trained(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model.to(DEVICE).eval()

    def forward(self, x, condition):
        return self.model(x, condition)


class Graph(nn.Module):
    """The client's graph, run by onnxruntime: the patches of a batch that share a frame's condition go in as one batch,
    as a frame's tiles do."""

    def __init__(self, session):
        super().__init__()
        self.session = session

    def forward(self, x, condition):
        x, condition = x.cpu().numpy(), condition.flatten(1).cpu().numpy()
        out = np.empty((x.shape[0], 3, x.shape[2] * SCALE, x.shape[3] * SCALE), dtype=np.float32)
        frames, which = np.unique(condition, axis=0, return_inverse=True)
        for index, frame in enumerate(frames):
            rows = np.flatnonzero(which.ravel() == index)
            out[rows] = run(self.session, x[rows], frame)
        return torch.from_numpy(out).to(DEVICE)


data = loader(val_set)
model_scores = training.evaluate(Trained(copy.deepcopy(fast)), data, trim)
graph_scores = training.evaluate(Graph(client_session), data, trim)
print(f"trained model: ssim-y {model_scores['ssim']:.6f}, luma {model_scores['y']:.4f} dB")
print(f"client graph:  ssim-y {graph_scores['ssim']:.6f}, luma {graph_scores['y']:.4f} dB")
assert abs(model_scores["ssim"] - graph_scores["ssim"]) < 1e-5

trained model: ssim-y 0.921249, luma 34.4659 dB
client graph:  ssim-y 0.921250, luma 34.4658 dB


## 8. Speed

The exported float16 graph at 16 tiles, timed on the 3090 (2026-11-13, the benchmark page, `mode=reuse`, 5 s) with
the told 24×2 in the same session. A second session in `mode=capture` timed them 1.41 and 6.95 ms - the same ratio,
and graph capture accepts the session, so no node falls back to the CPU. The float32 graph also runs on the page's
WebGL backend, which is the client's fallback.

In [13]:
# page ms, and kernels in us a tile by kind
FINAL_3090 = {"fast": (1.35, {"move": 24.8, "conv": 40.4, "matmul": 9.4, "add": 8.0}),
              "told 24x2": (6.65, {"move": 35.1, "conv": 325.9, "matmul": 22.6, "add": 26.0})}
print(f"{'':<12}{'3090 ms':>9}{'1660 Ti predicted':>19}{'conservative':>14}")
for name, (page, kernels) in FINAL_3090.items():
    central, conservative = predict(kernels)
    print(f"{name:<12}{page:9.2f}{central:19.1f}{conservative:14.1f}")
scaled = 34.6 * FINAL_3090["fast"][0] / FINAL_3090["told 24x2"][0]
print(f"the told 24x2 measured 34.6 ms on the 1660 Ti; scaled by the two graphs' 3090 times the fast one is {scaled:.1f} ms")

              3090 ms  1660 Ti predicted  conservative
fast             1.35                7.1           7.3
told 24x2        6.65               34.5          35.8
the told 24x2 measured 34.6 ms on the 1660 Ti; scaled by the two graphs' 3090 times the fast one is 7.0 ms


## 9. What it says

Run 2026-11-13 on the 9,628 validation patches; a model with two seeds is their mean (0.0001-0.0003 apart). The 1660
Ti column is predicted from the 3090 (section 2), central / conservative, from the candidate graphs timed in one session
(section 3); the told 24×2's was measured on the card.

| | SSIM-Y | luma dB | U / V dB | over bicubic | MAC / LR px | 1660 Ti, 16 tiles |
|---|---|---|---|---|---|---|
| bicubic | 0.9045 | 33.55 | 43.99 / 43.93 | | | |
| told 24×2 (ships) | 0.9289 | 35.07 | 44.03 / 43.97 | +0.0243 / +1.52 | ~13,000 | 34.6 ms, measured |
| **q24, frozen 4×4 skip (published)** | **0.9212** | **34.47** | **44.00 / 43.94** | **+0.0167 / +0.92** | 1,824 | **7.6 / 7.8** |
| q24, exact 6×6 skip | 0.9213 | 34.51 | 44.00 / 43.95 | +0.0167 / +0.96 | 1,824 | 8.7 / 8.9: over the bar |
| q24, 4×4 skip trained | 0.9214 | 34.43 | 43.04 / 43.23 | +0.0168 / +0.87 | 1,824 | 7.6 / 7.8 |
| q32, 4×4 skip trained | 0.9229 | 34.55 | 43.03 / 43.26 | +0.0184 / +1.00 | 3,072 | 8.2 / 9.1: over the bar |
| q20, 4×4 skip trained | 0.9197 | 34.34 | 43.06 / 43.22 | +0.0152 / +0.78 | 1,320 | 7.2 / 7.3 |

1. **q24 with a frozen skip fits, by prediction.** 7.6 ms central and 7.8 conservative in the candidates' session; the
   published graph timed again beside the told 24×2 (section 8) gives 7.1 and 7.3, and 7.0 scaled by the told graph's
   measured 34.6 ms. All of them are under the 8.3 ms bar and the 9.2 ms budget. With the 2.2-4.1 ms the client spends
   around a model, that is 9-12 ms a frame against 13.3: 60 fps at a 1280×720 stream, if the prediction holds. On the
   3090 it is 4.9× faster than the told 24×2.
2. **It keeps about two thirds of the told 24×2's gain** and all of bicubic's colour: +0.0167 SSIM-Y over bicubic
   against +0.0243 (69%), +0.92 dB luma against +1.52 (61%), U and V within 0.01 dB of bicubic. Three quarters of the
   gain on x264's frames (+0.0222 against +0.0302 on P frames), three fifths on NVENC's (+0.0111 against +0.0186).
3. **The skip has to stay frozen.** Trained with the rest under a loss that is 84% luma SSIM, the 4×4 skip gives up
   about 1 dB of U and V - colour worse than bicubic's - and buys nothing: the same SSIM-Y, 0.04 dB less luma. Frozen,
   it is within 0.04 dB of luma of the exact 6×6 skip, with the same colour, for 1.1 ms less on the 1660 Ti.
4. **Four channels of width are worth about 0.0016 SSIM-Y** (q20 0.9197, q24 0.9214, q32 0.9229, all with the skip
   trained, which does not move SSIM-Y). q32 is inside the 9.2 ms by the central prediction (8.2) and outside it by the
   conservative one (9.1). A frozen-skip q32 is the next model, if the card shows the room.
5. **Smaller does not pay.** About 3.1 ms of q24's time is the contract's fixed part - the casts, layout transposes,
   `DepthToSpace` and the skip - which no network size changes; q20 saves 0.2 ms and costs 0.0017 SSIM-Y.
6. **It is a drop-in.** The same inputs, output and halo as `upscale.onnx`. The client graph scores what the model
   scores (0.921250 SSIM-Y), a 1280×720 frame cut into the client's tiles matches the whole frame to 6e-7, the float16
   twin is within 0.16/255 of the float32 graph, and graph capture accepts it, so no node runs on the CPU.

**Not measured: anything on the GTX 1660 Ti.** Before calling it 60 fps there, time
`dist/upscale/upscale_fast_q24_frozen_skip_fp16_tile320x180_b16.onnx` as Run 8 of `bottlenecks_gtx1660ti.md` does (at
most 9.2 ms), then run the client with `upscale_fast.onnx` and `upscale_fast_fp16.onnx` in place of `upscale.onnx` and
`upscale_fp16.onnx` (`enhance.ms` at most 13.3). Its pictures have not been looked at either: the body works on 2×2
cells, and a faint pattern on that grid is something SSIM-Y need not show.

**Next:** measure it on the card and look at its output. If the card has room, train q32 with the frozen skip. The
client should choose between this graph and the told 24×2 by what the GPU affords rather than replace the told one
everywhere: the 3090 runs the told graph at 16 tiles in under 7 ms.